# 06 — FT-Transformer models

FT-Transformer classifier training and evaluation (A2: autoencoder-reduced, B2: pooled features). Uses `src.models.ft_transformer` and `src.training.evaluation`.

In [ ]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import logging
import numpy as np
import torch
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_reduced_dir, get_splits_dir, get_experiment_dir, get_experiment_output_dir, ensure_dir
from src.data import CLASS_NAMES, load_labels
from src.data.splits import load_splits
from src.models.ft_transformer import FTTransformer
from src.training.evaluation import compute_metrics, compute_per_class_metrics, save_metrics, save_confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# Toggles: which feature pipelines to run (set to True/False)
RUN_AUTOENCODER = False   # A2: autoencoder-reduced features
RUN_POOLED = True       # B2: pooled (kernel-informed) features

# FT-Transformer / training params (tuned for CPU: ~35k rows, max 256 features)
EPOCHS = 20
BATCH_SIZE = 1024   # larger batches = fewer steps/epoch (~34 steps for 35k)
LR = 2e-3           # scaled up for larger batch (was 1e-3 at batch 256)
WEIGHT_DECAY = 1e-4
D_TOKEN = 64        # smaller = faster per step, sufficient for 256-dim input
N_HEADS = 4         # must divide d_token
N_LAYERS = 2        # fewer layers = faster, often enough for tabular
DROPOUT = 0.1
NUM_CLASSES = len(CLASS_NAMES)
print("FT-Transformer: epochs=%s batch=%s lr=%s | d_token=%s n_heads=%s n_layers=%s | %d classes: %s" % (EPOCHS, BATCH_SIZE, LR, D_TOKEN, N_HEADS, N_LAYERS, NUM_CLASSES, CLASS_NAMES))
print("Run autoencoder (A2): %s | Run pooled (B2): %s" % (RUN_AUTOENCODER, RUN_POOLED))

FT-Transformer: epochs=20 batch=256 lr=0.001 | d_token=128 n_heads=8 n_layers=3 | 4 classes: ['AF', 'SVT', 'Sinus Brady', 'Sinus Rhythm']


In [ ]:
# Load splits and labels (same order as saved train/test/val .npy)
print("Loading splits and labels...")
idx_train, idx_val, idx_test = load_splits()
try:
    y_full = load_labels()
except FileNotFoundError:
    print("labels.npy not found; loading from raw dataset...")
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()
if y_full.ndim == 2:
    y_idx = np.argmax(y_full, axis=1)
else:
    y_idx = y_full
y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None
print("y_train %s y_test %s" % (y_train.shape, y_test.shape))

Loading splits and labels...
y_train (36120,) y_test (6773,)


In [ ]:
def train_ft_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, n_features, condition_name):
    """Train FT-Transformer on train set; evaluate on validation (if present) then on test. Save checkpoint and metrics."""
    set_global_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    exp_dir = ensure_dir(get_experiment_dir(condition_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(condition_name, checkpoints=True))

    model = FTTransformer(
        n_features=n_features,
        num_classes=NUM_CLASSES,
        d_token=D_TOKEN,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n_train = len(X_train)

    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0
        n_b = 0
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            bx = torch.from_numpy(X_train[idx]).float().to(device)
            by = torch.from_numpy(y_train[idx]).long().to(device)
            logits = model(bx)
            loss = torch.nn.functional.cross_entropy(logits, by)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_b += 1
        if (ep + 1) % 5 == 0 or ep == 0:
            print("  Epoch %d loss %.4f" % (ep + 1, total_loss / max(n_b, 1)))

    torch.save(model.state_dict(), ckpt_dir / "ft_transformer.pt")
    print("  Saved %s" % (ckpt_dir / "ft_transformer.pt"))

    model.eval()
    all_metrics = {}
    labels_list = list(range(NUM_CLASSES))
    with torch.no_grad():
        if X_val is not None and y_val is not None and len(y_val) > 0:
            logits_val = model(torch.from_numpy(X_val).float().to(device))
            y_pred_val = logits_val.argmax(dim=1).cpu().numpy()
            metrics_val = compute_metrics(y_val, y_pred_val, task="multiclass", labels=labels_list)
            all_metrics["weighted_f1_val"] = metrics_val["weighted_f1"]
            per_class_val = compute_per_class_metrics(y_val, y_pred_val, labels=labels_list, target_names=CLASS_NAMES)
            all_metrics["per_class_val"] = per_class_val
            print("  Validation weighted F1: %.4f" % metrics_val["weighted_f1"])
            print("  Validation per-class:")
            for p in per_class_val:
                print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
        logits_test = model(torch.from_numpy(X_test).float().to(device))
        y_pred_test = logits_test.argmax(dim=1).cpu().numpy()
    metrics_test = compute_metrics(y_test, y_pred_test, task="multiclass", labels=labels_list)
    per_class_test = compute_per_class_metrics(y_test, y_pred_test, labels=labels_list, target_names=CLASS_NAMES)
    all_metrics["weighted_f1"] = metrics_test["weighted_f1"]
    all_metrics["confusion_matrix"] = metrics_test.get("confusion_matrix", [])
    all_metrics["per_class"] = per_class_test
    save_metrics(all_metrics, exp_dir / "metrics.json")
    cm = np.array(metrics_test["confusion_matrix"])
    save_confusion_matrix(cm, exp_dir / "confusion_matrix.npy", path_png=exp_dir / "confusion_matrix.png", class_names=CLASS_NAMES)
    print("  Test weighted F1: %.4f" % metrics_test["weighted_f1"])
    print("  Test per-class:")
    for p in per_class_test:
        print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    return all_metrics


def print_metrics_summary(metrics, model_name):
    """Print test metrics for one model (weighted F1 + per-class table)."""
    print("\n" + "=" * 60)
    print("  %s — Test metrics" % model_name)
    print("=" * 60)
    print("  Weighted F1: %.4f" % metrics.get("weighted_f1", 0))
    if "weighted_f1_val" in metrics:
        print("  Weighted F1 (val): %.4f" % metrics["weighted_f1_val"])
    per = metrics.get("per_class", [])
    if per:
        print("  Per-class: %-12s %8s %8s %8s %8s" % ("Class", "Precision", "Recall", "F1", "Support"))
        print("  " + "-" * 52)
        for p in per:
            print("  %-12s %8.3f %8.3f %8.3f %8d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    print("=" * 60)

In [ ]:
# A2: Autoencoder + FT-Transformer (load autoencoder-reduced features)
metrics_a2 = None
if RUN_AUTOENCODER:
    print("=== A2: Autoencoder + FT-Transformer ===")
    red_dir = get_reduced_dir() / "autoencoder"
    X_train_a2 = np.load(red_dir / "train.npy").astype(np.float32)
    X_test_a2 = np.load(red_dir / "test.npy").astype(np.float32)
    if (red_dir / "val.npy").exists():
        X_val_a2 = np.load(red_dir / "val.npy").astype(np.float32)
    else:
        X_val_a2 = None
    n_features_a2 = X_train_a2.shape[1]
    print("Loaded autoencoder features: train %s test %s (n_features=%d)" % (X_train_a2.shape, X_test_a2.shape, n_features_a2))
    metrics_a2 = train_ft_and_evaluate(X_train_a2, X_val_a2, X_test_a2, y_train, y_val, y_test, n_features_a2, "A2_autoencoder_ft")
    print_metrics_summary(metrics_a2, "A2 (Autoencoder + FT-Transformer)")
else:
    print("Skipping A2 (autoencoder + FT-Transformer) — RUN_AUTOENCODER=False")

2026-02-28 13:55:35,287 | INFO | Global seed set to 0


=== A2: Autoencoder + FT-Transformer ===
Loaded autoencoder features: train (36120, 256) test (6773, 256) (n_features=256)
  Epoch 1 loss 1.0529
  Epoch 5 loss 1.0020
  Epoch 10 loss 0.9775
  Epoch 15 loss 0.9615
  Epoch 20 loss 0.9530
  Saved C:\Projects\DeepLearningProject\experiments\A2_autoencoder_ft\checkpoints\ft_transformer.pt
  Validation weighted F1: 0.5619
  Validation per-class:
    AF: P=0.643 R=0.697 F1=0.669 support=992
    SVT: P=0.000 R=0.000 F1=0.000 support=60
    Sinus Brady: P=0.581 R=0.737 F1=0.650 support=811
    Sinus Rhythm: P=0.353 R=0.137 F1=0.197 support=394


2026-02-28 19:51:33,659 | INFO | Saved metrics to C:\Projects\DeepLearningProject\experiments\A2_autoencoder_ft\metrics.json
2026-02-28 19:51:33,708 | INFO | Saved confusion matrix to C:\Projects\DeepLearningProject\experiments\A2_autoencoder_ft\confusion_matrix.npy
2026-02-28 19:51:34,259 | INFO | Saved confusion matrix figure to C:\Projects\DeepLearningProject\experiments\A2_autoencoder_ft\confusion_matrix.png


  Test weighted F1: 0.5770
  Test per-class:
    AF: P=0.652 R=0.711 F1=0.680 support=2969
    SVT: P=0.000 R=0.000 F1=0.000 support=162
    Sinus Brady: P=0.583 R=0.739 F1=0.652 support=2449
    Sinus Rhythm: P=0.460 R=0.167 F1=0.245 support=1193


NameError: name 'print_metrics_summary' is not defined

In [ ]:
# B2: Pooling + FT-Transformer (load pooled features)
metrics_b2 = None
if RUN_POOLED:
    print("=== B2: Pooling + FT-Transformer ===")
    red_dir = get_reduced_dir() / "pooled"
    X_train_b2 = np.load(red_dir / "train.npy").astype(np.float32)
    X_test_b2 = np.load(red_dir / "test.npy").astype(np.float32)
    if (red_dir / "val.npy").exists():
        X_val_b2 = np.load(red_dir / "val.npy").astype(np.float32)
    else:
        X_val_b2 = None
    n_features_b2 = X_train_b2.shape[1]
    print("Loaded pooled features: train %s test %s (n_features=%d)" % (X_train_b2.shape, X_test_b2.shape, n_features_b2))
    metrics_b2 = train_ft_and_evaluate(X_train_b2, X_val_b2, X_test_b2, y_train, y_val, y_test, n_features_b2, "B2_pooling_ft")
    print_metrics_summary(metrics_b2, "B2 (Pooling + FT-Transformer)")
else:
    print("Skipping B2 (pooling + FT-Transformer) — RUN_POOLED=False")

In [ ]:
# Summary: overall and per-class (test set)
print("=== Summary ===")
if metrics_a2 is not None:
    print("A2 (autoencoder + FT-Transformer) test weighted F1: %.4f" % metrics_a2["weighted_f1"])
if metrics_b2 is not None:
    print("B2 (pooling + FT-Transformer) test weighted F1: %.4f" % metrics_b2["weighted_f1"])
if metrics_a2 is not None or metrics_b2 is not None:
    print("\nPer-class (test set):")
    hdr = "%-12s | " % "Class" + " ".join("%8s" % h for h in (["A2_P", "A2_R", "A2_F1"] if metrics_a2 else []) + (["B2_P", "B2_R", "B2_F1"] if metrics_b2 else []))
    print(hdr)
    print("-" * len(hdr))
    for i, name in enumerate(CLASS_NAMES):
        row = [name]
        if metrics_a2:
            a2 = metrics_a2.get("per_class", [])[i] if "per_class" in metrics_a2 else {}
            row.extend([a2.get("precision", 0), a2.get("recall", 0), a2.get("f1", 0)])
        if metrics_b2:
            b2 = metrics_b2.get("per_class", [])[i] if "per_class" in metrics_b2 else {}
            row.extend([b2.get("precision", 0), b2.get("recall", 0), b2.get("f1", 0)])
        print("%-12s | " % row[0] + " ".join("%8.3f" % v for v in row[1:]))
else:
    print("No runs selected (set RUN_AUTOENCODER and/or RUN_POOLED to True).")